# Lexical Search using TF-IDF

## Sparse Term-Frequency Document Vectors for Research-Paper Abstracts

**Developer Role:** TF-IDF Developer (Lexical Search)

---

### Overview

This notebook implements **lexical representation** for research-paper search using **TF-IDF** (Term Frequency - Inverse Document Frequency), a classic sparse vector-space model.

TF-IDF has no notion of meaning: it only knows whether a word *literally* occurs in a document, weighted by how rare that word is across the whole corpus. Two abstracts describing the same idea with different vocabulary ("car" vs. "automobile") will **not** be considered similar by this model alone - that limitation is exactly what this arm of the case study is meant to measure, against Word2Vec's static semantic embeddings and BERT/SPECTER's contextual embeddings.

### What This Component Does

- Accepts a list of research-paper abstracts
- Generates a **TF-IDF document matrix** for the entire corpus
- Generates a **TF-IDF vector** for a new user query, using the SAME fitted vocabulary
- Returns embeddings as `scipy.sparse` matrices

### What This Component Does NOT Do

- Does **not** compute cosine similarity or any distance metric as part of the deliverable class
- Does **not** rank or retrieve documents
- Those responsibilities belong to a separate component of the case study
  (a small demo at the end of this notebook exercises them anyway, purely
  to show the embeddings work - it is explicitly **not** part of the class)

### Preprocessing Philosophy

Unlike BERT/SPECTER, TF-IDF relies entirely on **exact lexical matches between tokens**. That makes preprocessing far more important here than for a transformer model: every inflected/derived form of a word must be reduced to the same token, or the vectorizer treats "network", "networks" and "networking" as three unrelated, independently-weighted vocabulary columns. This notebook therefore performs the FULL classical NLP pipeline:

- Lowercasing
- Tokenization
- Removal of punctuation and standard English stop words
- POS-aware lemmatization (NLTK `WordNetLemmatizer`)

### Pipeline

```
Abstracts / Query
       |
Preprocessing (lowercase -> tokenize -> remove punctuation
               & stop words -> lemmatize)
       |
TfidfVectorizer (fit on the corpus, reused to transform queries)
       |
Sparse Embedding Vectors (scipy.sparse matrices)
```

### Position in the Overall Comparison

| Approach | Representation | Developer |
|----------|---------------|-----------|
| **TF-IDF** | **Lexical (exact word matching)** | **This notebook** |
| Word2Vec | Static semantic (fixed word vectors) | Developer 2 |
| SPECTER | Contextual scientific document embeddings | Developer 3 |


---
## Cell 1 — Install Dependencies

Install the required libraries:
- **scikit-learn**: Provides `TfidfVectorizer`, which builds the vocabulary and computes TF-IDF weights
- **nltk**: Tokenization, the English stop-word list, POS tagging and the WordNet lemmatizer
- **numpy**: Used to preview/inspect the sparse embeddings
- **pandas**: Loads the research-paper dataset from CSV

The `-q` flag suppresses verbose installation output.

In [1]:
!pip install -q scikit-learn nltk numpy pandas


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## Cell 2 — Imports

| Library | Purpose |
|---------|---------|
| `re`, `string` | Built-ins for basic cleaning (HTML/URLs) and the punctuation set |
| `numpy` | Used to preview/inspect the sparse embeddings |
| `pandas` | Loads the research-paper dataset from CSV |
| `nltk` | Tokenizer, English stop-word list, POS tagger, WordNet lemmatizer |
| `TfidfVectorizer` | Builds the vocabulary and computes TF-IDF weights (scikit-learn) |

In [2]:
# ============================================================
# Cell 2 — Imports
# ============================================================

import re                                        # Built-in: basic text cleaning (HTML tags, URLs, whitespace)
import string                                    # Built-in: the punctuation character set

import numpy as np                               # Used only to preview/inspect the sparse embeddings
import pandas as pd                              # Loads the dataset from CSV

import nltk                                      # Classical NLP toolkit (tokenizer, stop words, lemmatizer)
from nltk.corpus import stopwords, wordnet       # English stop-word list + WordNet POS constants
from nltk.stem import WordNetLemmatizer          # Rule/lexicon-based lemmatizer
from nltk.tokenize import word_tokenize          # Splits a sentence into a list of word tokens

from sklearn.feature_extraction.text import TfidfVectorizer  # Builds the TF-IDF vocabulary/matrix

print("All libraries imported successfully!")


All libraries imported successfully!


---
## Cell 3 — NLTK Resource Bootstrap

NLTK ships the *code*, but the *data* files (tokenizer tables, stop-word list, WordNet, POS tagger) are downloaded separately. This helper fetches anything missing, quietly, so the class works on a fresh machine.

(Kept identical to the Word2Vec developer's bootstrap so both classical-NLP arms of the case study share one preprocessing convention.)

In [3]:
# ============================================================
# Cell 3 — NLTK Resource Bootstrap
# ============================================================
# NLTK ships code, but the data files (tokenizer models, stop-word
# list, WordNet) must be downloaded once. This helper downloads any
# missing resource quietly so the class works on a fresh machine.
# ============================================================

# (resource path used by nltk.data.find, package name used by nltk.download)
_NLTK_RESOURCES = [
    ("tokenizers/punkt_tab", "punkt_tab"),                                  # sentence/word tokenizer tables
    ("tokenizers/punkt", "punkt"),                                          # tokenizer (older NLTK versions)
    ("corpora/stopwords", "stopwords"),                                     # English stop-word list
    ("corpora/wordnet", "wordnet"),                                         # lexical database used for lemmatization
    ("corpora/omw-1.4", "omw-1.4"),                                         # WordNet multilingual data (WordNet dependency)
    ("taggers/averaged_perceptron_tagger_eng", "averaged_perceptron_tagger_eng"),  # POS tagger for POS-aware lemmatization
]


def ensure_nltk_resources():
    """
    Download the NLTK data files required for preprocessing, if missing.

    Safe to call repeatedly: a resource that is already present on disk
    is detected by `nltk.data.find` and is not downloaded again.
    """
    for resource_path, package_name in _NLTK_RESOURCES:
        try:
            nltk.data.find(resource_path)
        except LookupError:
            nltk.download(package_name, quiet=True)


In [4]:
# Fetch the NLTK data now so the preprocessing cells below run cleanly
ensure_nltk_resources()

print("NLTK resources ready!")
print(f"English stop words available: {len(stopwords.words('english'))}")


NLTK resources ready!
English stop words available: 198


---
## Cell 4 — Define the `TFIDFSearcher` Class

This is the deliverable. Three things to note:

1. **`preprocess()`** performs the FULL classical pipeline (lowercase -> tokenize -> strip punctuation/stop words -> POS-aware lemmatize) - the opposite philosophy of the BERT/SPECTER arm, and deliberately so: TF-IDF can only match tokens that are spelled identically, so collapsing every inflected form onto one lemma is what makes the vocabulary usable at all.
2. **`preprocess` is passed straight to `TfidfVectorizer` as its `tokenizer`**, so scikit-learn calls our own function on every abstract during fitting *and* on every query during `.transform()` - guaranteeing corpus and query are always tokenized identically.
3. **Fitting happens lazily, exactly once.** Either `get_corpus_embeddings()` or `get_query_embedding()` can be called first; internally both funnel through a private `_ensure_fitted()` guard so the vocabulary/IDF weights are never accidentally re-fit (which would silently break query/corpus comparability).

In [5]:
# ============================================================
# Cell 4 — Define the TFIDFSearcher Class
# ============================================================

class TFIDFSearcher:
    """
    Lexical Search using TF-IDF (Term Frequency - Inverse Document Frequency).

    Unlike Word2Vec or BERT, this class has no concept of meaning: it
    represents each document as a sparse vector over the corpus
    vocabulary, weighted by how frequent a word is in that document
    versus how rare it is across the whole corpus. Only exact,
    lemma-matching words contribute to similarity, which is precisely
    the limitation this arm of the case study is meant to illustrate
    against the two semantic (Word2Vec / BERT) arms.

    Because TF-IDF only ever "sees" the tokens it was fit on, both the
    preprocessing AND the fitted vocabulary must be shared between the
    corpus and any later query - this class guarantees that by fitting
    ONE `TfidfVectorizer` on the corpus and reusing it (via `.transform`,
    never `.fit` again) for every subsequent query.

    Parameters
    ----------
    abstracts : list of str
        The research-paper abstracts to embed.
    max_features : int or None, default None
        If set, keeps only the top `max_features` terms by corpus
        frequency (caps the vocabulary/embedding dimension).
    ngram_range : tuple of (int, int), default (1, 1)
        Range of n-gram sizes to extract (e.g. (1, 2) keeps unigrams
        AND bigrams like "neural_network").
    sublinear_tf : bool, default True
        Replaces raw term frequency `tf` with `1 + log(tf)`, which
        stops very high-frequency words from dominating the vector -
        a standard TF-IDF refinement (Manning et al., IR textbook).

    Methods
    -------
    preprocess(text)
        Lowercase -> tokenize -> strip punctuation/stop words -> lemmatize.
        Returns a list of tokens (the vectorizer's custom tokenizer).
    get_corpus_embeddings()
        Returns a 2D SciPy sparse matrix of shape (n_papers, vector_size).
    get_query_embedding(query_string)
        Returns a 1x(vector_size) SciPy sparse row vector for the query.
    """

    def __init__(self, abstracts, max_features=None, ngram_range=(1, 1), sublinear_tf=True):
        """Store the corpus and configure (but do not yet fit) the vectorizer."""
        # ---- Store the raw corpus -------------------------------------
        self.abstracts = abstracts

        # ---- Set up the preprocessing tools ---------------------------
        # Downloaded once, then reused for every abstract and every query.
        ensure_nltk_resources()
        self.stop_words = set(stopwords.words("english"))   # e.g. "the", "of", "and"
        self.lemmatizer = WordNetLemmatizer()               # "networks" -> "network"
        self.punctuation = set(string.punctuation)          # ! " # $ % & ' ( ) * + , - . / ...

        # ---- Configure the TF-IDF vectorizer ---------------------------
        # `tokenizer=self.preprocess` means scikit-learn calls OUR
        # preprocessing function (lowercase/tokenize/stopword-strip/
        # lemmatize) on every document AND on every later query, so
        # both are guaranteed to be treated identically.
        #
        # `preprocessor=lambda text: text` and `lowercase=False` disable
        # scikit-learn's OWN default cleaning step, since `preprocess`
        # already does all of that itself. `token_pattern=None` silences
        # scikit-learn's warning that `token_pattern` is unused when a
        # custom `tokenizer` is supplied.
        self.vectorizer = TfidfVectorizer(
            tokenizer=self.preprocess,
            preprocessor=lambda text: text,
            lowercase=False,
            token_pattern=None,
            max_features=max_features,
            ngram_range=ngram_range,
            sublinear_tf=sublinear_tf,
        )

        # Cache for get_corpus_embeddings(): fit_transform() is only
        # ever run once, on first use, by either public method below.
        self._corpus_embeddings = None
        self.vector_size = None

        print(f"TFIDFSearcher created for {len(self.abstracts)} abstracts "
              f"(vectorizer not yet fit - happens lazily on first use).")

    # --------------------------------------------------------------
    # Preprocessing
    # --------------------------------------------------------------

    @staticmethod
    def _wordnet_pos(treebank_tag):
        """
        Map a Penn-Treebank POS tag to the POS constant WordNet expects.

        The lemmatizer needs to know the part of speech to be accurate:
        without it, "learning" (verb) stays "learning" and "was" stays
        "wa". Defaults to NOUN, which is WordNet's own default.
        """
        if treebank_tag.startswith("J"):
            return wordnet.ADJ      # adjective
        if treebank_tag.startswith("V"):
            return wordnet.VERB     # verb
        if treebank_tag.startswith("R"):
            return wordnet.ADV      # adverb
        return wordnet.NOUN         # noun (default)

    def preprocess(self, text):
        """
        Turn a raw string into a clean list of lemmatized content words.

        This is passed directly to `TfidfVectorizer` as its `tokenizer`,
        so it is called on every abstract during fitting AND on every
        query during `.transform()` - the exact same function, every time.

        Steps
        -----
        0. Basic cleaning : strip HTML tags, URLs and stray whitespace.
        1. Lowercasing    : "Neural" and "neural" become the same token.
        2. Tokenization   : split the sentence into a list of words.
        3. Filtering      : drop punctuation, digits, single/double
                            characters and standard English stop words.
        4. Lemmatization  : reduce inflected forms to their base form
                            ("networks" -> "network", "learned" -> "learn")
                            so all variants share one TF-IDF column.

        Parameters
        ----------
        text : str
            Raw abstract or query text.

        Returns
        -------
        list of str
            The cleaned, lemmatized tokens.
        """
        # Guard against NaN / non-string input coming from a CSV column
        if not isinstance(text, str):
            return []

        # --- Step 0: basic cleaning ---------------------------------
        text = re.sub(r"<[^>]+>", " ", text)          # remove HTML tags: <br>, <p>, </div>
        text = re.sub(r"http\S+|www\.\S+", " ", text) # remove URLs
        text = re.sub(r"\s+", " ", text).strip()      # collapse repeated whitespace

        # --- Step 1: lowercase --------------------------------------
        text = text.lower()

        # --- Step 2: tokenize ---------------------------------------
        tokens = word_tokenize(text)

        # --- Step 3: remove punctuation, numbers and stop words -----
        cleaned_tokens = []
        for token in tokens:
            # Strip punctuation attached to a word ("state-of-the-art" -> "stateoftheart",
            # "model." -> "model"). Tokens that were pure punctuation become "".
            token = "".join(char for char in token if char not in self.punctuation)

            if not token.isalpha():        # drops "", "2021", "3d", stray symbols
                continue
            if len(token) < 3:             # drops noise like "et", "al", "e"
                continue
            if token in self.stop_words:   # drops "the", "we", "of", "which", ...
                continue

            cleaned_tokens.append(token)

        if not cleaned_tokens:
            return []

        # --- Step 4: POS-aware lemmatization ------------------------
        # POS-tag first so the lemmatizer knows whether "training" is a
        # noun or a verb, then reduce each token to its dictionary form.
        tagged_tokens = nltk.pos_tag(cleaned_tokens)
        lemmatized_tokens = [
            self.lemmatizer.lemmatize(token, self._wordnet_pos(tag))
            for token, tag in tagged_tokens
        ]

        return lemmatized_tokens

    # --------------------------------------------------------------
    # Internal: fit the vectorizer exactly once
    # --------------------------------------------------------------

    def _ensure_fitted(self):
        """
        Fit the TfidfVectorizer on the corpus, if it has not been fit yet.

        This is called automatically by BOTH `get_corpus_embeddings()`
        and `get_query_embedding()`, so either method can be called
        first and the vectorizer is still only ever fit ONCE (fitting
        twice would silently change the vocabulary/IDF weights and
        break query/corpus comparability).
        """
        if self._corpus_embeddings is not None:
            return  # already fit - nothing to do

        print(f"Fitting TF-IDF vectorizer on {len(self.abstracts)} abstracts "
              f"(preprocessing happens inside the vectorizer)...")

        # fit_transform() runs `preprocess` on every abstract, builds the
        # vocabulary/IDF weights from the result, and returns the corpus
        # matrix in one pass - a second, separate transform pass would be
        # wasted work.
        self._corpus_embeddings = self.vectorizer.fit_transform(self.abstracts)
        self.vector_size = len(self.vectorizer.vocabulary_)

        print(f"TFIDFSearcher fitted with {len(self.abstracts)} abstracts.")
        print(f"Vocabulary size:     {self.vector_size}")
        print(f"Embedding dimension: {self.vector_size}  (one dimension per vocabulary term)")

    # --------------------------------------------------------------
    # Public API
    # --------------------------------------------------------------

    def get_corpus_embeddings(self):
        """
        Generate TF-IDF document vectors for ALL abstracts.

        Pipeline:
            Raw abstracts -> preprocess (tokenizer) -> TfidfVectorizer.fit_transform -> sparse matrix

        The vectorizer is fit only once; repeated calls return the
        cached matrix.

        Returns
        -------
        scipy.sparse.csr_matrix
            A 2D SPARSE matrix of shape (n_papers, vector_size).
            Example: 727 abstracts over a vocabulary of ~6700 terms ->
            (727, 6700), with the vast majority of entries being 0.0.

        Note
        ----
        Unlike Word2Vec/BERT (which return dense NumPy arrays), TF-IDF
        vectors are naturally sparse - most terms in the vocabulary do
        not appear in any single abstract. `scipy.sparse` matrices work
        directly with `sklearn.metrics.pairwise.cosine_similarity`, so
        no conversion is required downstream. Call `.toarray()` on the
        result if a dense NumPy array is ever needed instead.
        """
        self._ensure_fitted()
        return self._corpus_embeddings

    def get_query_embedding(self, query_string):
        """
        Generate a TF-IDF vector for a single user query.

        The query is tokenized with the EXACT SAME `preprocess` function
        and projected onto the EXACT SAME (already-fitted) vocabulary as
        the corpus - `.transform()` is used, never `.fit()`, so a query
        can never introduce new vocabulary/IDF weights that the corpus
        vectors don't also have.

        Pipeline:
            Query -> preprocess (tokenizer) -> TfidfVectorizer.transform -> sparse row vector

        Parameters
        ----------
        query_string : str
            The user's natural-language search query.

        Returns
        -------
        scipy.sparse.csr_matrix
            A SPARSE row vector of shape (1, vector_size). A query whose
            words are all out-of-vocabulary returns an all-zero row.
        """
        # Fitting must have happened already (or happens now, on first
        # use) - a query can only be projected onto a vocabulary that
        # already exists.
        self._ensure_fitted()

        # `.transform` (not `.fit_transform`) reuses the corpus vocabulary/
        # IDF weights untouched, keeping corpus and query comparable.
        return self.vectorizer.transform([query_string])


print("TFIDFSearcher class defined successfully!")


TFIDFSearcher class defined successfully!


---
## Cell 5 — Load the Dataset

Load the research-paper abstracts from `abstract_sentences.csv` (provided by the Scholar Inbox authors).

**Important:** The CSV contains multiple rows per paper (one per annotated sentence). We use `drop_duplicates()` on the `abstract` column to get one entry per unique abstract. This guarantees the order of papers stays **exactly the same** for TF-IDF, Word2Vec, and BERT implementations.

**For Google Colab:** Upload `abstract_sentences.csv` to the Colab runtime, or mount Google Drive and adjust the path accordingly.

In [6]:
# ============================================================
# Cell 5 — Load the Dataset
# ============================================================
# The dataset "abstract_sentences.csv" is provided by the
# Scholar Inbox authors. Each row contains an abstract with
# sentence-level annotations. The same abstract appears
# multiple times, so we use drop_duplicates() to get one
# entry per unique paper.
#
# drop_duplicates() preserves the original order, ensuring
# consistency across TF-IDF, Word2Vec, and BERT implementations.
# ============================================================

# Load the dataset provided by the Scholar Inbox authors
df = pd.read_csv("../abstract_sentences.csv")

# Extract the 'abstract' column and drop duplicates
# drop_duplicates() guarantees that the order of papers stays EXACTLY the same
# for TF-IDF, Word2Vec, and BERT.
unique_abstracts = df['abstract'].drop_duplicates().dropna().tolist()

print(f"Successfully loaded {len(unique_abstracts)} unique research papers!")

# ---------------------------------------------------------
# 'unique_abstracts' is now a standard Python list of strings.
# You can now pass this list into your TF-IDF, Word2Vec, or BERT code!
# ---------------------------------------------------------

# Preview the first abstract (truncated for display)
print(f"\nExample abstract (first 300 characters):")
print(f"{unique_abstracts[0][:300]}...")


Successfully loaded 727 unique research papers!

Example abstract (first 300 characters):
We present an efficient method for joint optimization of topology, materials and lighting from multi-view image observations. Unlike recent multi-view reconstruction approaches, which typically produce entangled 3D representations encoded in neural networks, we output triangle meshes with spatially-...


---
## Cell 6 — Initialize the `TFIDFSearcher`

Create an instance of `TFIDFSearcher` by passing the list of unique abstracts. Fitting the vectorizer is deferred - it happens automatically the first time `get_corpus_embeddings()` or `get_query_embedding()` is called (Cell 8).

In [7]:
# ============================================================
# Cell 6 — Initialize the TFIDFSearcher
# ============================================================
# Pass the unique abstracts to create the searcher instance.
# Fitting the vectorizer happens lazily on first use.
# ============================================================

searcher = TFIDFSearcher(unique_abstracts)


TFIDFSearcher created for 727 abstracts (vectorizer not yet fit - happens lazily on first use).


---
## Cell 7 — Preprocessing Walk-through

Before fitting anything, let's see exactly what `preprocess()` does to a real abstract. This is the step that matters most for TF-IDF - since it only ever matches identical tokens, everything hinges on getting this right.

In [8]:
# ============================================================
# Cell 7 — Preprocessing Walk-through
# ============================================================

sample_raw = unique_abstracts[0]
sample_tokens = searcher.preprocess(sample_raw)

print("BEFORE (raw abstract, first 300 characters):")
print(f"  {sample_raw[:300]}...")
print()
print(f"AFTER (preprocessed tokens, {len(sample_tokens)} total):")
print(f"  {sample_tokens}")


BEFORE (raw abstract, first 300 characters):
  We present an efficient method for joint optimization of topology, materials and lighting from multi-view image observations. Unlike recent multi-view reconstruction approaches, which typically produce entangled 3D representations encoded in neural networks, we output triangle meshes with spatially-...

AFTER (preprocessed tokens, 93 total):
  ['present', 'efficient', 'method', 'joint', 'optimization', 'topology', 'material', 'light', 'multiview', 'image', 'observation', 'unlike', 'recent', 'multiview', 'reconstruction', 'approach', 'typically', 'produce', 'entangled', 'representation', 'encode', 'neural', 'network', 'output', 'triangle', 'mesh', 'spatiallyvarying', 'material', 'environment', 'light', 'deploy', 'traditional', 'graphic', 'engine', 'unmodified', 'leverage', 'recent', 'work', 'differentiable', 'render', 'coordinatebased', 'network', 'compactly', 'represent', 'volumetric', 'texturing', 'alongside', 'differentiable', 'march', '

---
## Cell 8 — Generate Corpus Embeddings

This is the first call that actually fits the `TfidfVectorizer`: every abstract is preprocessed by our own `preprocess()` (called internally as the vectorizer's `tokenizer`), the vocabulary and IDF weights are learned from the result, and the fitted corpus matrix is returned.

**Expected output shape:** `(727, vocab_size)` - a SPARSE matrix, not a dense array.

In [9]:
# ============================================================
# Cell 8 — Generate Corpus Embeddings
# ============================================================
# This fits the TF-IDF vocabulary on ALL abstracts and returns
# their sparse document vectors.
#
# Expected output: a 2D SciPy sparse matrix
#   - Rows    = number of unique abstracts (727)
#   - Columns = TF-IDF vocabulary size (corpus-dependent)
# ============================================================

corpus_embeddings = searcher.get_corpus_embeddings()

print(f"\nCorpus embeddings shape: {corpus_embeddings.shape}")


Fitting TF-IDF vectorizer on 727 abstracts (preprocessing happens inside the vectorizer)...
TFIDFSearcher fitted with 727 abstracts.
Vocabulary size:     6721
Embedding dimension: 6721  (one dimension per vocabulary term)

Corpus embeddings shape: (727, 6721)


---
## Cell 9 — Generate a Query Embedding

The query passes through the **same** `preprocess()` and is projected onto the **same** fitted vocabulary as the abstracts (`.transform`, never `.fit`), so query and corpus vectors share one vector space and their columns line up term-for-term.

We reuse the exact same demo query as the Word2Vec and BERT notebooks, so all three arms can later be compared side by side on identical input.

In [10]:
# ============================================================
# Cell 9 — Generate Query Embedding
# ============================================================
# The query is processed through the SAME preprocessing and the
# SAME fitted vocabulary as the corpus, so both embeddings live
# in the same TF-IDF vector space (i.e. share the same columns).
# ============================================================

# Example query — a natural-language search for research papers
query = "AI methods for identifying malicious network activity"

query_embedding = searcher.get_query_embedding(query)

print(f"Query: '{query}'")
print(f"Query embedding shape: {query_embedding.shape}")


Query: 'AI methods for identifying malicious network activity'
Query embedding shape: (1, 6721)


---
## Cell 10 — Inspect the Embeddings

Verify the generated embeddings are well-formed and that corpus and query share one vector space. **No similarity or ranking is performed here** - that is outside the scope of this component.

In [11]:
# ============================================================
# Cell 10 — Inspect Embeddings
# ============================================================
# Verify the generated embeddings are correct.
# No similarity calculation or ranking is performed here.
# ============================================================

print("=" * 60)
print("CORPUS EMBEDDINGS")
print("=" * 60)
print(f"Type:            {type(corpus_embeddings)}")
print(f"Data type:       {corpus_embeddings.dtype}")
print(f"Shape:           {corpus_embeddings.shape}")
print(f"  -> {corpus_embeddings.shape[0]} documents, each represented as a {corpus_embeddings.shape[1]}-dimensional sparse vector")
print(f"Non-zero entries: {corpus_embeddings.nnz} / {corpus_embeddings.shape[0] * corpus_embeddings.shape[1]} "
      f"({100 * corpus_embeddings.nnz / (corpus_embeddings.shape[0] * corpus_embeddings.shape[1]):.3f}% dense)")
print(f"\nFirst document embedding (first 10 non-zero terms):")
first_row = corpus_embeddings[0].tocoo()
terms = searcher.vectorizer.get_feature_names_out()
top_terms = sorted(zip(first_row.col, first_row.data), key=lambda x: -x[1])[:10]
for col, weight in top_terms:
    print(f"  {terms[col]:<20s} {weight:.4f}")

print()
print("=" * 60)
print("QUERY EMBEDDING")
print("=" * 60)
print(f"Type:            {type(query_embedding)}")
print(f"Data type:       {query_embedding.dtype}")
print(f"Shape:           {query_embedding.shape}")
print(f"  -> 1 query, represented as a {query_embedding.shape[1]}-dimensional sparse vector")
print(f"Non-zero entries: {query_embedding.nnz}")
print(f"\nQuery embedding (non-zero terms):")
query_row = query_embedding.tocoo()
for col, weight in sorted(zip(query_row.col, query_row.data), key=lambda x: -x[1]):
    print(f"  {terms[col]:<20s} {weight:.4f}")

print()
print("=" * 60)
print("VERIFICATION")
print("=" * 60)
print(f"Corpus and query embeddings have the same dimension: "
      f"{corpus_embeddings.shape[1] == query_embedding.shape[1]} "
      f"({corpus_embeddings.shape[1]} == {query_embedding.shape[1]})")
print(f"\n-> Both embeddings exist in the same {corpus_embeddings.shape[1]}-dimensional vector space.")
print(f"-> They can be compared using cosine similarity or other metrics")
print(f"   (handled by a separate component).")


CORPUS EMBEDDINGS
Type:            <class 'scipy.sparse._csr.csr_matrix'>
Data type:       float64
Shape:           (727, 6721)
  -> 727 documents, each represented as a 6721-dimensional sparse vector
Non-zero entries: 56124 / 4886167 (1.149% dense)

First document embedding (first 10 non-zero terms):
  material             0.2131
  light                0.1956
  differentiable       0.1902
  entangled            0.1602
  tetrahedron          0.1602
  allfrequency         0.1602
  trianglebased        0.1602
  rasterizers          0.1602
  mesh                 0.1565
  unmodified           0.1508

QUERY EMBEDDING
Type:            <class 'scipy.sparse._csr.csr_matrix'>
Data type:       float64
Shape:           (1, 6721)
  -> 1 query, represented as a 6721-dimensional sparse vector
Non-zero entries: 5

Query embedding (non-zero terms):
  malicious            0.6848
  activity             0.5604
  identify             0.4006
  network              0.1880
  method               0.1453

VERI

---
## Cell 11 — What Did the Vectorizer Actually Learn?

A quick look at the **IDF weights** makes the "lexical, frequency-based" nature of TF-IDF concrete: a HIGH IDF term is rare across the corpus (appears in very few abstracts, so it is very discriminative when it does appear); a LOW IDF term is common across almost every abstract (so it barely helps distinguish one paper from another, even after stop-word removal).

In [12]:
# ============================================================
# Cell 11 — Most / Least Distinctive Terms (by IDF weight)
# ============================================================

idf = searcher.vectorizer.idf_
terms = searcher.vectorizer.get_feature_names_out()
order = np.argsort(idf)

print("LOWEST-IDF terms (most common across the corpus, least distinctive):")
for i in order[:15]:
    print(f"  {terms[i]:<20s} idf={idf[i]:.4f}")

print()
print("HIGHEST-IDF terms (rarest across the corpus, most distinctive):")
for i in order[::-1][:15]:
    print(f"  {terms[i]:<20s} idf={idf[i]:.4f}")


LOWEST-IDF terms (most common across the corpus, least distinctive):
  method               idf=1.4634
  propose              idf=1.4766
  model                idf=1.5058
  use                  idf=1.5988
  image                idf=1.6687
  show                 idf=1.7732
  learn                idf=1.8537
  approach             idf=1.8537
  network              idf=1.8932
  result               idf=1.9273
  demonstrate          idf=1.9956
  novel                idf=2.0068
  train                idf=2.0648
  task                 idf=2.0728
  stateoftheart        idf=2.0769

HIGHEST-IDF terms (rarest across the corpus, most distinctive):
  aachen               idf=6.8972
  abundance            idf=6.8972
  abusive              idf=6.8972
  accessory            idf=6.8972
  accident             idf=6.8972
  worstcase            idf=6.8972
  worth                idf=6.8972
  wpre                 idf=6.8972
  wrist                idf=6.8972
  wrongattributed      idf=6.8972
  wrt           

---
## Summary

| | |
|---|---|
| **Model** | TF-IDF (scikit-learn `TfidfVectorizer`), fit on the corpus |
| **Preprocessing** | lowercase -> tokenize -> remove punctuation/numbers/stop words -> POS-aware lemmatize |
| **Document strategy** | Sparse term-frequency / inverse-document-frequency weighting (no averaging, no neural model) |
| **Corpus output** | `get_corpus_embeddings()` -> sparse matrix `(n_papers, vocab_size)` |
| **Query output** | `get_query_embedding(q)` -> sparse row vector `(1, vocab_size)` |

**Strengths:** fast, fully interpretable (every dimension is a literal word, with a human-readable weight), needs no training/GPU, and is very strong at exact-terminology lookups - e.g. finding a missed citation that uses the same jargon as the query.

**Limitations (what the Word2Vec and BERT arms address):**
- **No notion of meaning** - a query and an abstract that describe the same idea in different words score close to zero if they share no lemmas.
- **High dimensionality** - one dimension per vocabulary term (thousands), almost entirely zeros per document.
- **No word order or context** - "model predicts the image" and "image predicts the model" produce the same bag-of-words vector.
- **Sensitive to preprocessing choices** - a missed lemmatization case silently splits one concept into two never-matching columns.

### Position in the Overall Comparison

```
Research Paper Abstracts
          |
          +------------------+------------------+
          |                  |                  |
          v                  v                  v
       TF-IDF            Word2Vec           SPECTER
       Lexical        Static Semantic    Contextual Semantic
          |                  |                  |
          v                  v                  v
   Document Vectors    Document Vectors   Document Vectors  <- YOU ARE HERE
          |                  |                  |
          +------------------+------------------+
                             |
                             v
                    Compare Search Quality
```


---
# Demo Run — Lexical Search over the Dataset

Everything above is the deliverable. The cells below are a **demonstration** that the embeddings behave the way TF-IDF is expected to - including its characteristic *weakness*.

The ranking logic here (cosine similarity + `argsort`) is intentionally written **outside** the `TFIDFSearcher` class, since the comparison/ranking component belongs to a different part of the case study.

---
## Cell 12 — A Small Search Helper (demo only)

Cosine similarity measures the **angle** between two sparse vectors, ignoring their magnitude - the standard way to compare TF-IDF vectors, and exactly what `scikit-learn`'s `cosine_similarity` does with sparse input directly (no `.toarray()` needed).

In [13]:
# ============================================================
# Cell 12 — Search Helper (DEMO ONLY — not part of the class)
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity


def search(query_string, top_k=5, preview_chars=260):
    """Rank abstracts by cosine similarity to the query and print the top-k."""
    # 1. Embed the query with the SAME searcher (same fitted vocabulary)
    q_vec = searcher.get_query_embedding(query_string)

    # 2. Cosine similarity against every document vector (sparse-safe)
    scores = cosine_similarity(q_vec, corpus_embeddings)[0]

    # 3. Take the top-k highest scores (descending)
    top_indices = np.argsort(scores)[::-1][:top_k]

    print("=" * 78)
    print(f"QUERY: {query_string}")
    print(f"Preprocessed tokens: {searcher.preprocess(query_string)}")
    print("=" * 78)
    for rank, idx in enumerate(top_indices, start=1):
        print(f"\n[{rank}] score = {scores[idx]:.4f}   (paper index {idx})")
        print(f"    {unique_abstracts[idx][:preview_chars].strip()}...")
    print()
    return top_indices, scores


print("Search helper ready!")


Search helper ready!


---
## Cell 13 — Run the Searches

Three queries, chosen deliberately to show BOTH sides of TF-IDF:

1. A query built from **the same vocabulary** as its target abstract -> should score well.
2. The canonical cross-notebook query (also used by the Word2Vec/BERT demos) -> a realistic case.
3. A **paraphrase** of a topic in the corpus that shares almost no literal words with it -> should score poorly, demonstrating the vocabulary-mismatch weakness that motivates the Word2Vec/BERT arms.

In [14]:
# ============================================================
# Cell 13 — Demo Searches
# ============================================================

demo_queries = [
    "differentiable rendering of triangle meshes with material and lighting optimization",  # lexical overlap expected
    "AI methods for identifying malicious network activity",                                 # canonical cross-notebook query
    "teaching a computer to understand pictures without any labels",                         # paraphrase, low literal overlap
]

for q in demo_queries:
    search(q, top_k=3)
    print()


QUERY: differentiable rendering of triangle meshes with material and lighting optimization
Preprocessed tokens: ['differentiable', 'render', 'triangle', 'mesh', 'material', 'light', 'optimization']

[1] score = 0.4100   (paper index 0)
    We present an efficient method for joint optimization of topology, materials and lighting from multi-view image observations. Unlike recent multi-view reconstruction approaches, which typically produce entangled 3D representations encoded in neural networks, w...

[2] score = 0.3481   (paper index 590)
    We propose a neural inverse rendering pipeline called IRON that operates on photometric images and outputs high-quality 3D content in the format of triangle meshes and material textures readily deployable in existing graphics pipelines. Our method adopts neura...

[3] score = 0.2830   (paper index 534)
    We present a differentiable rendering framework for material and lighting estimation from multi-view images and a reconstructed geometry. In the

---
## Cell 14 — Sample Input -> Output Demo

A complete walkthrough with one abstract and one query, showing exactly what goes in and what comes out - using the same sample abstract/query as the BERT notebook's demo, so the two can be compared directly.

In [15]:
# ============================================================
# Cell 14 — Sample Input → Output Demo
# ============================================================
# A complete walkthrough with one abstract and one query
# showing exactly what goes in and what comes out.
# ============================================================

# ── SAMPLE INPUT ─────────────────────────────────────────────
sample_abstract = (
    "We propose a novel deep learning framework for detecting "
    "cyber intrusions in network traffic. Our method combines "
    "convolutional neural networks with attention mechanisms to "
    "identify malicious patterns in packet-level data. Experiments "
    "on the CICIDS2017 benchmark demonstrate that our approach "
    "achieves 98.7% detection accuracy while maintaining low "
    "false positive rates, outperforming traditional machine "
    "learning baselines such as Random Forest and SVM."
)

sample_query = "AI methods for identifying malicious network activity"

# ── PRINT INPUT ──────────────────────────────────────────────
print("=" * 70)
print("                        INPUT")
print("=" * 70)
print()
print("SAMPLE ABSTRACT:")
print(f'  "{sample_abstract}"')
print()
print("SAMPLE QUERY:")
print(f'  "{sample_query}"')
print()

# ── GENERATE EMBEDDINGS ──────────────────────────────────────
# Create a mini searcher with just the one sample abstract
demo_searcher = TFIDFSearcher([sample_abstract])

# Generate the document embedding (1 abstract → 1 sparse vector)
demo_doc_embedding = demo_searcher.get_corpus_embeddings()

# Generate the query embedding
demo_query_embedding = demo_searcher.get_query_embedding(sample_query)

# ── PRINT OUTPUT ─────────────────────────────────────────────
print()
print("=" * 70)
print("                        OUTPUT")
print("=" * 70)
print()
demo_terms = demo_searcher.vectorizer.get_feature_names_out()
print("DOCUMENT EMBEDDING:")
print(f"  Shape : {demo_doc_embedding.shape}")
print(f"  Type  : {type(demo_doc_embedding)}")
print(f"  Non-zero terms: {demo_doc_embedding.nnz}")
doc_row = demo_doc_embedding.tocoo()
for col, weight in sorted(zip(doc_row.col, doc_row.data), key=lambda x: -x[1])[:8]:
    print(f"    {demo_terms[col]:<15s} {weight:.4f}")
print(f"  ↑ {demo_doc_embedding.shape[1]} vocabulary dimensions represent this single abstract (mostly zero)")
print()
print("QUERY EMBEDDING:")
print(f"  Shape : {demo_query_embedding.shape}")
print(f"  Type  : {type(demo_query_embedding)}")
print(f"  Non-zero terms: {demo_query_embedding.nnz}")
q_row = demo_query_embedding.tocoo()
for col, weight in sorted(zip(q_row.col, q_row.data), key=lambda x: -x[1]):
    print(f"    {demo_terms[col]:<15s} {weight:.4f}")
print()
print("=" * 70)
print("                     WHAT THIS MEANS")
print("=" * 70)
print()
print(f"  • The abstract has been converted into a {demo_doc_embedding.shape[1]}-dimensional sparse vector.")
print(f"  • The query   has been converted into a {demo_query_embedding.shape[1]}-dimensional sparse vector.")
print(f"  • Both vectors live in the SAME {demo_doc_embedding.shape[1]}-dimensional space (this mini corpus's vocabulary).")
print(f"  • A separate component can now compute cosine similarity")
print(f"    between them to determine how relevant the paper is to the query.")


                        INPUT

SAMPLE ABSTRACT:
  "We propose a novel deep learning framework for detecting cyber intrusions in network traffic. Our method combines convolutional neural networks with attention mechanisms to identify malicious patterns in packet-level data. Experiments on the CICIDS2017 benchmark demonstrate that our approach achieves 98.7% detection accuracy while maintaining low false positive rates, outperforming traditional machine learning baselines such as Random Forest and SVM."

SAMPLE QUERY:
  "AI methods for identifying malicious network activity"

TFIDFSearcher created for 1 abstracts (vectorizer not yet fit - happens lazily on first use).
Fitting TF-IDF vectorizer on 1 abstracts (preprocessing happens inside the vectorizer)...
TFIDFSearcher fitted with 1 abstracts.
Vocabulary size:     41
Embedding dimension: 41  (one dimension per vocabulary term)

                        OUTPUT

DOCUMENT EMBEDDING:
  Shape : (1, 41)
  Type  : <class 'scipy.sparse._csr.csr_